# Normalization of the dataset
Calculate the values required for a Z Normalization of the images and heigth profiles are determined and stored in a JSON file for later usage in the model

In [40]:
from pathlib import Path
import os
import json

import numpy as np
import torch
import torchvision.io
from torchvision.transforms import v2

In [41]:
DATA_PATH = Path("../rawdata/processed")
BEFORE_PATH = "before"
AFTER_PATH = "after"
IMAGE_PATH = "centered-image"
HEIGHT_PATH = "centered-height"
MASK_PATH = "mask"

In [42]:
scale = v2.ToDtype(torch.float32, scale=True)

In [45]:
def get_image_as_numpy(image_path, mask):
    img = scale(torchvision.io.read_image(image_path)).detach().numpy()
    return np.ma.array(img, mask=mask)

def get_height_as_numpy(height_path, mask):
    return np.ma.array(np.load(height_path), mask=mask)

def get_mask_as_numpy(mask_path, data_type=None):
    "Load mask"
    mask = np.load(mask_path)
    if data_type=="image":
        mask = np.repeat(mask[np.newaxis, :, :], 3, axis=0)
    return mask

def calc_image_stats(img):
    "Calculate number of elements, sum of elements and sum of elements squared per channel"
    return (img.count(axis=(1, 2)), img.sum(axis=(1, 2)), (img*img).sum(axis=(1, 2)))

def calc_height_stats(height):
    "Calculate number of elements, sum of elements and sum of elements squared"
    return (np.array([height.count()]), np.array([height.sum()]), np.array([(height*height).sum()]))

def get_sample_ids(file_path):
    return [i[:-4] for i in os.listdir(file_path)]

def calc_std(count, sum, sum_squared):
    return np.sqrt(count * sum_squared - sum**2)/count

def calc_mean(count, sum):
    return sum/count

def calc_statistics(timed_data_path, data_type):
    if data_type == "image":
        file_path = timed_data_path / IMAGE_PATH
    elif data_type == "height":
        file_path = timed_data_path / HEIGHT_PATH
    else:
        raise ValueError

    sample_ids = get_sample_ids(file_path)
    total_count = 0
    total_sum = 0
    total_sum_sqrd = 0

    for id in sample_ids:
        mask = get_mask_as_numpy(timed_data_path / MASK_PATH / f"{id}.npy", data_type=data_type)

        if data_type == "image":
            image = get_image_as_numpy(timed_data_path / IMAGE_PATH / f"{id}.png", mask)
            count, sum, sum_squared = calc_image_stats(image)
        elif data_type == "height":
            height = get_height_as_numpy(timed_data_path / HEIGHT_PATH / f"{id}.npy", mask)
            count, sum, sum_squared = calc_height_stats(height)
        else:
            raise ValueError

        total_count += count
        total_sum += sum
        total_sum_sqrd += sum_squared

    return len(sample_ids), calc_mean(total_count, total_sum), calc_std(total_count, total_sum, total_sum_sqrd)

In [46]:
results = {}

for time in [BEFORE_PATH, AFTER_PATH]:
    data_dict = {}
    for data_type in ["image", "height"]:
        # Calculate statistics across all files of the time and data_type
        n, mean, std = calc_statistics(DATA_PATH / time, data_type)
        print(f"The {time} {data_type} statistics for {n+1} files are mean: {mean} and std: {std}")
        data_dict[data_type] = {"mean": list(mean), "std": list(std)}
    results[time] = data_dict

# Write statistics to json file
with open(DATA_PATH / "normalization.json", mode="w") as write_file:
    json.dump(results, write_file)
# json.dumps(results)

The before image statistics for 843 files are mean: [0.4401761705230707 0.41535370178470726 0.38807271359315026] and std: [0.19227372712931012 0.18292735683525096 0.1749953446886863]
The before height statistics for 843 files are mean: [0.00294193] and std: [0.01225754]
The after image statistics for 843 files are mean: [0.5172880908795748 0.4931250645152486 0.4669963566049725] and std: [0.1616452774890432 0.15893740548488636 0.16303375293936567]
The after height statistics for 843 files are mean: [-0.00413525] and std: [0.01288719]


In [59]:
def check_statistics(time, data_type):
    """Check if calculating the standard z score of every single data set leads to mean zero and std 1"""
    if data_type == "image":
        file_path = DATA_PATH / time / IMAGE_PATH
    elif data_type == "height":
        file_path = DATA_PATH / time / HEIGHT_PATH
    else:
        raise ValueError

    # Load the statistics from json
    with open(DATA_PATH / "normalization.json", mode="r") as write_file:
        results = json.load(write_file)


    sample_ids = get_sample_ids(file_path)
    total_count = 0
    total_sum = 0
    total_sum_sqrd = 0

    for id in sample_ids:
        mask = get_mask_as_numpy(DATA_PATH / time / MASK_PATH / f"{id}.npy", data_type=data_type)

        if data_type == "image":
            image = get_image_as_numpy(DATA_PATH / time / IMAGE_PATH / f"{id}.png", mask)
            image = (image - np.array(results[time][data_type]["mean"])[:, np.newaxis, np.newaxis]) / np.array(results[time][data_type]["std"])[:, np.newaxis, np.newaxis]
            count, sum, sum_squared = calc_image_stats(image)
        elif data_type == "height":
            height = get_height_as_numpy(DATA_PATH / time / HEIGHT_PATH / f"{id}.npy", mask)
            height = (height - np.array(results[time][data_type]["mean"])) / np.array(results[time][data_type]["std"])
            count, sum, sum_squared = calc_height_stats(height)
        else:
            raise ValueError

        total_count += count
        total_sum += sum
        total_sum_sqrd += sum_squared

    return len(sample_ids), calc_mean(total_count, total_sum), calc_std(total_count, total_sum, total_sum_sqrd)

In [ ]:
for time in [BEFORE_PATH, AFTER_PATH]:
    data_dict = {}
    for data_type in ["image", "height"]:
        # Calculate statistics across all files of the time and data_type
        n, mean, std = check_statistics(time, data_type)
        print(f"The {time} {data_type} statistics for {n+1} files are mean: {mean} and std: {std}")
        data_dict[data_type] = {"mean": list(mean), "std": list(std)}

The before image statistics for 843 files are mean: [-0.001185694260016466 -0.0016956024685465779 -0.0022641428859857707] and std: [1.0031152190635484 1.0042454877979106 1.005405978272342]
The before height statistics for 843 files are mean: [1.02368336e-16] and std: [1.]
The after image statistics for 843 files are mean: [0.0001969964155569191 -0.00019791549637826665 -0.0007819067332595538] and std: [0.9993521991702458 1.0007859723049293 1.0024590002105]
The after height statistics for 843 files are mean: [-5.51990033e-17] and std: [1.]


'{"before": {"image": {"mean": [0.4401761705230707, 0.41535370178470726, 0.38807271359315026], "std": [0.19227372712931012, 0.18292735683525096, 0.1749953446886863]}, "height": {"mean": [0.002941930208161772], "std": [0.012257536351815233]}}, "after": {"image": {"mean": [0.5172880908795748, 0.4931250645152486, 0.4669963566049725], "std": [0.1616452774890432, 0.15893740548488636, 0.16303375293936567]}, "height": {"mean": [-0.004135247325878172], "std": [0.01288719196302128]}}}'